# Pipeline ETL — Zika/SINAN (2024–2026)

**Fluxo:** CSV → Pandas → transformação e esquema estrela em memória → MySQL.

O contrato é o mesmo nos dois pipelines: cinco dimensões e `fato_notificacao_zika`.
Consulte o [dicionário](../docs/dicionario_de_dados_e_mapeamento_de_divergencias.md),
o [DDL MySQL](../sql/01_script_ddl_do_esquema_estrela.sql) e o [diagrama](../docs/diagrama_fluxo_integracao.png).

Execute todas as células em ordem. A publicação substitui os dados das seis tabelas estrela
no banco configurado, em uma transação. Use um banco dedicado ao projeto. As análises e gráficos
ficam em [analises_zika.ipynb](analises_zika.ipynb).

## Ambiente

Python 3.10–3.12, MySQL 8.0.16+ e dependências de `requirements.txt`.
No Colab, clone o repositório, entre na raiz com `%cd` e instale as dependências
com `%pip install -r requirements.txt`. O servidor MySQL deve estar acessível pelo Colab.


In [ ]:
from pathlib import Path
import sys

# Execute na raiz do clone ou em notebooks/ (inclusive no Colab).
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'pipelines/contrato.json').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Clone o repositório e abra o notebook dentro dele.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
import os
# Crie previamente um banco dedicado ao projeto, usando utf8mb4.
# Configure a conexão antes de executar. A senha vem do ambiente, sem ser salva no notebook.
os.environ.setdefault('MYSQL_HOST', '127.0.0.1')
os.environ.setdefault('MYSQL_PORT', '3306')
os.environ.setdefault('MYSQL_USER', 'root')
os.environ.setdefault('MYSQL_DATABASE', 'zika_dw')
# os.environ['MYSQL_PASSWORD'] = getpass.getpass('Senha MySQL: ')


## Extract — leitura dos três CSVs

In [ ]:
from pipelines.core import extract, transform, star
raw = extract()
print(f"Linhas brutas: {len(raw):,}")

## Transform — filtragem direta do DataFrame e modelagem em Python

Deduplicação exata, validação de datas, decodificação de domínios, métricas e dimensões acontecem em memória.

In [ ]:
rows, rejected, audit = transform(raw)
tables = star(rows)
print(audit)
assert audit["brutas"] == audit["duplicadas_exatas"] + audit["rejeitadas"] + audit["validas"]

## Load — esquema estrela no MySQL

O DDL preserva PKs, FKs, tipos e restrições. Falhas na carga causam rollback.

In [ ]:
from pipelines.database import connect, publish, validate
with connect() as connection:
    publish(connection, tables)
    counts = validate(connection)
assert counts["fato_notificacao_zika"] == len(rows)
print(counts)